In [1]:
import pickle
from pathlib import Path

import pandas as pd

# Reading the atmodeller output (pickle) file

In [2]:
def read_from_pickle(file_path: Path | str) -> dict[str, pd.DataFrame]:
    """Reads Atmodeller output from a pickle file

    See also:
        https://docs.python.org/3/library/pickle.html

    Args:
        file_path: A pickle file to read

    Returns:
        A dictionary of Atmodeller output data
    """
    with open(file_path, "rb") as handle:
        atmodeller_data: dict[str, pd.DataFrame] = pickle.load(handle)

    return atmodeller_data

Update to the location of the pickle file you want to read. The example file below is generated when you run the trappist.ipynb notebook in this directory.

In [3]:
file_path = "t1e_280K.pkl"
data: dict[str, pd.DataFrame] = read_from_pickle(file_path)

Get some basic properties of the output data

In [4]:
# Get available dataframe names
print("Available data = ", data.keys())

# Get data for a species. Gas species are _g, liquid species _l, and condensed species _cr
# following the JANAF naming convention. Rows are each a unique simulation and columns are all the
# outputs (mass, moles, number, pressure, etc.)
print("Columns in dataframe = ", data["CO2_g"].columns)

Available data =  dict_keys(['H2_g', 'H2O_g', 'O2_g', 'CO_g', 'CO2_g', 'CH4_g', 'N2_g', 'H3N_g', 'S2_g', 'H2S_g', 'O2S_g', 'OS_g', 'Cl2_g', 'ClH_g', 'C_cd', 'H2O_cd', 'S_cd', 'ClH4N_cd', 'element_C', 'element_Cl', 'element_H', 'element_N', 'element_O', 'element_S', 'state', 'raw', 'gas', 'constraints', 'residual', 'solver'])
Columns in dataframe =  Index(['gas_number', 'gas_mass', 'gas_number_density', 'dissolved_number',
       'dissolved_mass', 'total_number', 'total_mass', 'molar_mass',
       'volume_mixing_ratio', 'gas_mass_fraction', 'pressure', 'fugacity',
       'fugacity_coefficient', 'dissolved_ppmw'],
      dtype='object')


Now we can get some outputs and perform some post-processing.

In [7]:
# Do some basic stats
CO2_g_pressure = data["CO2_g"]["pressure"]
print("Mean CO2_g pressure = ", CO2_g_pressure.median())

# Let's look at how much liquid water we have
H2O_l_mass = data["H2O_cd"]["total_mass"]
print("Mean H2O_l mass = ", H2O_l_mass.median())

# Let's find all models with at least 0.2 Earth oceans of water
OCEAN_MASS_H2O = 1.3851863627795307e21
mask = H2O_l_mass >= OCEAN_MASS_H2O * 0.2

H2O_data_1EO = data["H2O_cd"][mask]
num_models = len(H2O_data_1EO)
total_models = len(data["H2O_cd"])
percent_models = num_models / total_models * 100
print(f"{num_models} ({percent_models}%) models have more than 0.2 Earth oceans of water")

# Let's get some element totals
C_element_total_mass = data["element_C"]["total_mass"]
print("C total mass = ", C_element_total_mass)

Mean CO2_g pressure =  49.37267911391615
Mean H2O_l mass =  4.5085841100314107e+18
5 (5.0%) models have more than 0.2 Earth oceans of water
C total mass =  0     4.443793e+21
1     2.766864e+20
2     1.689706e+21
3     3.296342e+20
4     1.553165e+20
          ...     
95    1.393840e+20
96    3.959171e+19
97    2.890716e+19
98    8.917150e+19
99    1.564862e+19
Name: total_mass, Length: 100, dtype: float64


In [8]:
# Remember that dictionary entries are dataframes

element_C = data["element_C"]
print("type(element_C) = ", type(element_C))

# Which means columns are series

C_element_total_mass = data["element_C"]["total_mass"]
print("type(C_element_total_mass)C = ", type(C_element_total_mass))

type(element_C) =  <class 'pandas.core.frame.DataFrame'>
type(C_element_total_mass)C =  <class 'pandas.core.series.Series'>
